In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import seaborn as sns
import torch
import torchvision.transforms.v2 as T

from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm

sns.set_theme()

In [ ]:
# необходимая трансформация данных - датасет возвращает объекты PIL.Image, а нам нужно работать с векторами признаков
data_transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float64, scale=True),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)),
    T.Lambda(lambda x: x.flatten()),
])

trainset = CIFAR10(root=".", train=True, transform=data_transform, download=True)
testset = CIFAR10(root=".", train=False, transform=data_transform)

In [ ]:
# функция визуализации выученных весов нейронной сети для каждого класса
def visualize_weights(weights_matrix: npt.NDArray[np.float64], class_names: list[str]):
    weights_matrix = weights_matrix.copy()
    # превратим веса относительно каждого класса в набор картинок 32x32x3
    weights_matrix = weights_matrix.reshape(3, 32, 32, 10)
    # переставим каналы в правильном порядке
    weights_matrix = np.moveaxis(weights_matrix, 0, -2)
    # для нормировки узнаем минимальный и максимальный веса
    w_min, w_max = weights_matrix.min(), weights_matrix.max()
    fig, axes = plt.subplots(2, 5, figsize=(16, 8))
    axes = [ax for ax_row in axes for ax in ax_row]
    for idx, ax in enumerate(axes):
        # отмасштабируем веса в отрезок [0, 255]
        w_img = 255 * (weights_matrix[:, :, :, idx].squeeze() - w_min) / (w_max - w_min)
        w_img = w_img.astype("uint8")
        ax.imshow(w_img)
        ax.grid(False)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        ax.set_title(class_names[idx])
    plt.show()

Остальные функции для обучения сети возьмите из семинара.

# Задание 1

Вам нужно провести эксперименты по обучению однослойной сети. Постарайтесь получить максимальное качество (accuracy) на тестовой выборке. Экспериментируйте с количеством эпох, batch_size, шагом обучения и коэффициентом регуляризации (reg, дробное число >= 0). По итогам экспериментов зафиксируйте параметры с которыми получилось наилучшее качество и соответствующие графики. Попробуйте описать, как каждый из изменяемых параметров влияет на процесс обучения.

*Хороший результат для однослойной сети - ~45% accuracy*

In [ ]:
model = ...
history = ...

In [ ]:
# график метрик
plot_metrics(history)

In [ ]:
# финальные метрики на валидации
_, test_acc, test_report = eval_model(model, testset, batch_size=128)
print(f"Accuracy: {test_acc:.2%}")
print(test_report)

*ваш текст здесь*

# Задание 2

Возьмите свою лучшую модель из первого задания и визуализируйте ее веса с помощью функции ниже. Попробуйте предположить, что на них можно увидеть.

In [ ]:
visualize_weights(model.W, class_names=trainset.classes)

*ваш текст здесь*

# Задание 3: Двухслойная сеть

Дополните код в классе двухслойной сети по аналогии с однослойной сетью (для справки используйте материалы лекций и код однослойной сети выше). В качестве функции активации скрытого слоя используйте ReLU. Признаком правильной реализации является постепенное уменьшение train_loss по мере обучения.

In [ ]:
class TwoLayersNet:
    def __init__(self, in_dim: int, h_dim: int, out_dim: int) -> None:
        # in_dim - размерность входа сети, out_dim - размерность выхода сети (количество классов)
        # h_dim - размерность скрытого слоя сети
        # случайная инициализация весов сети
        rng = np.random.default_rng()
        self.W1 = 1e-3 * rng.normal(size=(in_d, h_d))
        self.bias1 = np.zeros(h_d)
        self.W2 = 1e-3 * rng.normal(size=(h_d, out_d))
        self.bias2 = np.zeros(out_d)
        self.n_weights = self.W1.size + self.bias1.size + self.W2.size + self.bias2.size

    def __repr__(self) -> str:
        return f"TwoLayersNet(n_weights = {self.n_weights})"

    def forward(self, X_batch: npt.NDArray[np.float64]) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
        # вычисляет ответ сети для пакета данных
        # в качестве функции активации скрытого слоя используйте ReLU (подсказка - вам может помочь функция np.maximum)
        # h - активации скрытого слоя, они нужны методу backward() для правильного расчета градиента
        # ваш код здесь
        out1 = ... + self.bias1
        # h = ReLU, примененная к out1 - выход первого слоя сети
        h = ...
        # умножаем h на соответствующие веса и прибавляем bias
        out2 = ... + self.bias2
        return out2, h

    def predict_proba(self, X_batch: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        # возвращает вероятности классов для каждого входящего примера
        logits, _ = self.forward(X_batch)
        return softmax(logits)

    def backward(self, X_batch: npt.NDArray[np.float64], y_batch: npt.NDArray[np.float64], reg: float) -> tuple[float, dict[str, npt.NDArray[np.float64]]]:
        # метод рассчитывает значение функции ошибки и ее градиент по весам для применения в градиентном спуске
        # получаем ответы сети на данном шаге и активации скрытого слоя
        logits, hidden = self.forward(X_batch)
        y_pred = softmax(logits)
        # в качестве функции ошибки для задачи классификации используем кросс-энтропию
        loss = ce_loss(y_batch, y_pred) + 0.5 * reg * (np.sum(self.W1 ** 2) + np.sum(self.W2 ** 2))
        # расчет градиента выводится с помощью математики для данного конкретного типа сети
        # градиенты для каждой матрицы весов положим в словарь
        gradients: dict[str, npt.NDArray[np.float64]] = {}
        batch_size = X_batch.shape[0]
        y_pred[range(batch_size), y_batch] -= 1
        gradient_w2 = hidden.T @ y_pred
        gradients["W2"] = gradient_w2 / batch_size + reg * self.W2
        gradients["bias2"] = np.mean(y_pred, axis=0)

        dh = y_pred @ self.W2.T
        dh_relu = (hidden > 0) * dh
        gradient_w1 = X_batch.T @ dh_relu
        gradients["W1"] = gradient_w1 / batch_size + reg * self.W1
        gradients["bias1"] = np.mean(dh_relu, axis=0)
        return loss, gradients

    def optimize_step(self, X_batch: npt.NDArray[np.float64], y_batch: npt.NDArray[np.float64], lr: float, reg: float) -> float:
        # обучение сети на одном батче, градиентный спуск
        loss, gradients = self.backward(X_batch, y_batch, reg)
        # итерация градиентного спуска
        # ваш код здесь
        self.W1 = ...
        self.bias1 = ...
        self.W2 = ...
        self.bias2 = ...
        return loss

# Задание 4

Теперь проведите эксприменты с двухслойной сетью. У вас появляется новый параметр - количество нейронов в скрытом слое. Замечание: параметры, подобранные для однослойной сети не будут оптимальными для двухслойной. Если ваша сеть сильно переобучается - либо уменьшайте количество нейронов, либо повышайте параметр reg, если сеть практически не обучается - попробуйте увеличить learning rate или уменьшить reg.

Опишите результаты экспериментов и зафиксируйте параметры и графики лучшей модели.

In [ ]:
model = ...
history = ...

In [ ]:
plot_metrics(history)

*Для хорошо обученной двухслойной сети можно получить accuracy > 50%. При этом двухслойная сеть может сходиться сильно медленнее однослойной.*

*ваш текст здесь*